# How much slower the Python implementation is

This tool removes manual line breaks from Markdown prose. It has two
implementations, one in Python and one in Rust, and for every input the shared
conformance corpus covers they produce identical bytes. The only difference
between them is how long they take to run. This notebook measures that
difference.

There is no single number for it. Three costs make up the total, and they
differ from each other. Starting the program costs a fixed amount, once per
run. Handling one more file adds a small amount for each file. Rewriting the
text inside a file costs more as the file holds more text.

Which of the three accounts for most of the total depends on how the tool is
invoked. For a pre-commit hook on one edited file, starting the program is
almost all of it. For a run over every Markdown file in a repository, the
per-file cost is. For a run over one long document, rewriting the text is.

Every number and every chart below is computed by the code cell above it, so
nothing in this text can contradict the output next to it. A number written
into a sentence would be wrong the next time the notebook runs.

In [1]:
import platform
import re
import subprocess
import sys
from pathlib import Path


def git(*argv: str) -> str:
    """Return the stripped stdout of a git command, or ``''`` if it failed."""
    return subprocess.run(
        ['git', *argv], capture_output=True, text=True, check=False
    ).stdout.strip()


REPO = Path(git('rev-parse', '--show-toplevel'))
PYTHON_CLI = REPO / '.venv/bin/unwrap-markdown-prose-py'
RUST_CLI = REPO / 'target/release/unwrap-markdown-prose-rs'

for path in (PYTHON_CLI, RUST_CLI):
    if not path.exists():
        raise SystemExit(f'missing {path}: run `uv sync` and `cargo build --release`')

# The optimization level is read from Cargo.toml, not copied into this notebook.
# The release profile is set to optimize for binary size, so every Rust timing
# below is the timing of a size-optimized build. Anyone comparing against their
# own build needs to know which setting produced these numbers.
level = re.search(
    r'^\s*opt-level\s*=\s*(\S+)', (REPO / 'Cargo.toml').read_text(), re.MULTILINE
)
OPT_LEVEL = level.group(1).strip('"') if level else 'default'

# Publishing the prebuilt binaries is triggered by a tag. Several rows of the
# final recommendation depend on whether a release exists, so that is read here
# rather than assumed. Local tags are used because this notebook does not make
# network requests, and a tag is what starts the release either way.
TAGS = git('tag', '--list', 'v*').split()

rustc = subprocess.run(
    ['rustc', '--version'], capture_output=True, text=True, check=False
).stdout.strip()

print(f'machine   {platform.machine()}  {platform.system()} {platform.release()}')
print(f'python    {platform.python_version()}')
print(f'rust      {rustc}')
print(f'revision  {git("rev-parse", "--short", "HEAD")}')
BINARY_KB = RUST_CLI.stat().st_size / 1024
tuning = ', optimized for size rather than speed' if OPT_LEVEL in {'z', 's'} else ''
releases = ', '.join(TAGS) if TAGS else 'none yet; see the last cell'

print(f'binary    {BINARY_KB:.0f} KB at opt-level {OPT_LEVEL!r}{tuning}')
print(f'releases  {releases}')

machine   arm64  Darwin 23.5.0
python    3.10.18
rust      rustc 1.86.0 (05f9846f8 2025-03-31)
revision  d0619cf
binary    344 KB at opt-level 'z', optimized for size rather than speed
releases  none yet; see the last cell


## Method

Each measurement times one complete run of the program, because that is what a
pre-commit hook or a CI step actually costs. Three warm-up runs are discarded,
and the runs after them are timed.

**Both the minimum and the median are reported.** The minimum is the better
estimate of what the program costs, because it is the run least affected by
other work happening on the machine at the same time. The median is printed
next to it so that the difference between the two is visible. A large
difference means the machine was busy, and a cell below checks for this and
reports it.

**The cost of starting a process is measured, not subtracted.** Every number
here includes the time to start a process: the operating system call that
creates it, the one that loads the program, and the pipe setup that this
notebook's own timing loop adds. That time is very small compared to the Python
total, and large compared to the Rust total. Dividing one by the other
therefore understates how much faster Rust is.

Subtracting that time would be less accurate than reporting it. The next cell
times three programs that do almost nothing, and they differ from each other by
more than the entire Rust runtime, so there is no single correct value to
subtract. Instead the cheapest of them is reported as a floor, drawn on the
charts, and used to mark any ratio whose smaller number is close to that floor.
Those ratios are labeled as lower bounds: the real difference is larger than
the number shown.

**The two implementations are compared inside the benchmark.** For every run,
the timing loop records a hash of the output and the exit code, then compares
both between Python and Rust. Without this check, a program that rejected its
arguments and exited immediately would record a very fast time and look like an
improvement. Recording a hash for every run, rather than keeping only the last
one, also detects a file being modified while the benchmark is reading it.

In [2]:
import hashlib
import statistics
import time
from dataclasses import dataclass

WARMUP, REPS = 3, 30

# How far above the minimum the median may be, as a percentage, before a
# measurement is reported as unreliable. Taken from the startup rows, which are
# the most repeatable measurements here and are a few percent apart on an
# otherwise idle machine.
SPREAD_LIMIT = 15.0

# How close to the floor the smaller of two times may be before their ratio is
# reported as a lower bound instead of a value.
FLOOR_FACTOR = 2.0

# Marks a ratio as a lower bound. Defined here as a name because an escape
# sequence cannot appear inside an f-string expression before Python 3.12.
BOUND = '\u2265'


@dataclass(frozen=True, slots=True)
class Timing:
    """Every sample from one measured command, and what those runs returned."""

    samples: tuple[float, ...]
    codes: frozenset[int]
    # One hash per repetition, rather than one saved copy of the output. If two
    # runs of the same command return different bytes, something changed the
    # files while the benchmark was reading them. Saving only one copy of the
    # output cannot show that, because the copy saved is the last run.
    digests: frozenset[str]

    @property
    def min(self) -> float:
        """Return the run least affected by other work on the machine."""
        return min(self.samples)

    @property
    def median(self) -> float:
        """Return the middle sample, which is raised by a busy machine."""
        return statistics.median(self.samples)

    @property
    def spread(self) -> float:
        """Return how far the median is above the minimum, as a percentage."""
        return (self.median - self.min) / self.min * 100


def measure(argv: list[str], reps: int = REPS) -> Timing:
    """Time ``reps`` complete runs of ``argv``, after ``WARMUP`` discarded runs."""
    for _ in range(WARMUP):
        subprocess.run(argv, capture_output=True, check=False)
    samples, codes, digests = [], set(), set()
    for _ in range(reps):
        started = time.perf_counter()
        done = subprocess.run(argv, capture_output=True, check=False)
        samples.append((time.perf_counter() - started) * 1000)
        codes.add(done.returncode)
        digests.add(hashlib.sha256(done.stdout).hexdigest())
    return Timing(tuple(samples), frozenset(codes), frozenset(digests))


# Three programs that do almost nothing, timed through the same loop as
# everything else. The cheapest is used as the floor. All three are printed
# because the differences between them are the reason none of them is
# subtracted from the measurements.
REFERENCES = [['/bin/echo', 'x'], ['/usr/bin/true'], ['/usr/bin/printf', '']]

reference = {argv[0]: measure(argv) for argv in REFERENCES if Path(argv[0]).exists()}
FLOOR = min(reference.values(), key=lambda timing: timing.min)
SPAWN_FLOOR = FLOOR.min

for name, timing in sorted(reference.items(), key=lambda item: item[1].min):
    print(f'{name:<18} {timing.min:>6.2f} ms   (median {timing.median:>5.2f} ms)')
spread = max(t.min for t in reference.values()) - SPAWN_FLOOR
print(f'\nfloor              {SPAWN_FLOOR:>6.2f} ms, the cheapest of the three')
print(f'they differ by     {spread:>6.2f} ms, which is why none of them is subtracted')

/usr/bin/true        1.10 ms   (median  1.15 ms)
/bin/echo            1.25 ms   (median  1.31 ms)
/usr/bin/printf      1.25 ms   (median  1.29 ms)

floor                1.10 ms, the cheapest of the three
they differ by       0.16 ms, which is why none of them is subtracted


In [3]:
import json

scratch = Path('/tmp/markdown-prose-bench')
scratch.mkdir(exist_ok=True)

# One paragraph broken across several lines is the unit of work this tool
# exists to undo.
PARAGRAPH = 'A paragraph that has been hard\nwrapped across three\nseparate lines.\n\n'
(scratch / 'tiny.md').write_text(PARAGRAPH)

tracked = subprocess.run(
    ['git', 'ls-files', '*.md'], cwd=REPO, capture_output=True, text=True, check=True
).stdout.split()
(scratch / 'tracked.txt').write_text(
    '\n'.join(str(REPO / name) for name in tracked) + '\n'
)

# A generated set of files, so that the number of files can be varied
# independently of this repository.
bulk = scratch / 'bulk'
bulk.mkdir(exist_ok=True)
for index in range(2000):
    (bulk / f'{index}.md').write_text(PARAGRAPH * 3)

# One large document. Startup is a negligible part of the time taken to process
# it, so what remains is the speed of the transform itself.
large = scratch / 'large.md'
large.write_text(PARAGRAPH * 60000)

# Passed explicitly because this notebook runs from the docs directory, and the
# tool looks for .unwrapignore in the directory it is run from. Without it the
# benchmark would process the corpus directory, which holds test fixtures
# rather than prose: around 200 expected-output files of a few dozen bytes
# each, plus one file that is deliberately not valid UTF-8 and that the tool
# correctly refuses to read. Those files are not this repository's Markdown,
# and an average taken over them is not this repository's file size.
IGNORE = ['--ignore-file', str(REPO / '.unwrapignore')]

# Which files count as in scope is determined by running the tool rather than
# reimplemented here, because the tool defines the ignore rules.
scope = json.loads(
    subprocess.run(
        [
            str(PYTHON_CLI),
            '--files-from',
            str(scratch / 'tracked.txt'),
            *IGNORE,
            '--json',
        ],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
)
in_scope = [Path(entry['path']) for entry in scope['files']]
sizes = sorted(path.stat().st_size for path in in_scope)
REPO_MEAN_BYTES = sum(sizes) / len(sizes)

print(
    f'{len(tracked)} tracked Markdown files, {len(in_scope)} in scope and '
    f'{len(tracked) - len(in_scope)} excluded as test fixtures'
)
average = f'mean {REPO_MEAN_BYTES:,.0f} bytes per file'
print(f'  in scope: {sum(sizes) / 1024:.0f} KB, {average}')
print(f'generated file: {len(PARAGRAPH) * 3} bytes')
print(f'large document: {large.stat().st_size / 1e6:.1f} MB')

219 tracked Markdown files, 6 in scope and 213 excluded as test fixtures
  in scope: 178 KB, mean 30,395 bytes per file
generated file: 207 bytes
large document: 4.1 MB


In [4]:
from IPython.display import Markdown, display


@dataclass(frozen=True, slots=True)
class Scenario:
    """One row of the table below: what was processed, and how long each took."""

    label: str
    python: Timing
    rust: Timing

    @property
    def timings(self) -> tuple[tuple[str, Timing], tuple[str, Timing]]:
        """Return each timing with the name of the program that produced it."""
        return ('python', self.python), ('rust', self.rust)


def file_list(count: int) -> list[str]:
    """Return arguments naming the first ``count`` of the generated files."""
    listing = scratch / f'bulk-{count}.txt'
    listing.write_text(
        '\n'.join(str(bulk / f'{index}.md') for index in range(count)) + '\n'
    )
    return ['--files-from', str(listing)]


# One run over the 4 MB document takes about a second in Python, so that row
# uses fewer repetitions than the others. Stated here rather than left for a
# reader to work out from how long the cell takes.
LARGE_REPS = 10

scenarios = [
    ('one file', [str(scratch / 'tiny.md')], REPS),
    ('a typical commit: 5 files', file_list(5), REPS),
    (
        f'this repository: {len(in_scope)} prose files at '
        f'{REPO_MEAN_BYTES / 1024:.0f} KB each',
        ['--files-from', str(scratch / 'tracked.txt'), *IGNORE],
        REPS,
    ),
    ('a large repository: 2000 files', file_list(2000), REPS),
    (f'one {large.stat().st_size / 1e6:.0f} MB document', [str(large)], LARGE_REPS),
]

results: list[Scenario] = []
for label, args, reps in scenarios:
    results.append(
        Scenario(
            label=label,
            python=measure([str(PYTHON_CLI), *args, '--json'], reps),
            rust=measure([str(RUST_CLI), *args, '--json'], reps),
        )
    )


def ratio_cell(row: Scenario) -> str:
    """Return Python divided by Rust, marked as a bound when Rust is near the floor."""
    value = row.python.min / row.rust.min
    mark = BOUND if row.rust.min < SPAWN_FLOOR * FLOOR_FACTOR else ''
    return f'{mark}{value:.1f}x'


table = [
    '| what is being processed | Python min / median | Rust min / median | Python is |',
    '| -- | --: | --: | --: |',
]
for row in results:
    table.append(
        f'| {row.label} '
        f'| {row.python.min:.1f} / {row.python.median:.1f} ms '
        f'| {row.rust.min:.1f} / {row.rust.median:.1f} ms '
        f'| {ratio_cell(row)} slower |'
    )
table.append('')
table.append(
    f'{BOUND} means the Rust time is within {FLOOR_FACTOR:.0f} times the '
    f'{SPAWN_FLOOR:.2f} ms floor, so most of what it measures is the cost of starting '
    'any process at all. Those ratios are lower bounds, and the real difference is '
    'larger.'
)
display(Markdown(chr(10).join(table)))

# Three separate questions, because they have three separate answers, and only
# the first of them can invalidate a recommendation.
#
#   AGREE    did the two implementations return the same bytes and the same
#            exit code? This is the assumption the whole document depends on.
#   STEADY   did each command return the same result on every repetition? It
#            may not, because the file list comes from the working directory,
#            and a file edited while the benchmark runs appears here as a read
#            error in one repetition out of thirty.
#   CLEAN    did every run exit with status 0? A consistent non-zero status
#            means the tool reported a file it could not read, which is a fact
#            about the files rather than about either implementation.
AGREE = all(
    row.python.digests == row.rust.digests and row.python.codes == row.rust.codes
    for row in results
)
unsteady = [
    (row.label, name)
    for row in results
    for name, timing in row.timings
    if len(timing.digests) > 1 or len(timing.codes) > 1
]
CLEAN = all(
    timing.codes == frozenset({0}) for row in results for _, timing in row.timings
)
noisy = [
    (row.label, name, timing.spread)
    for row in results
    for name, timing in row.timings
    if timing.spread > SPREAD_LIMIT
]

if AGREE:
    print('Both implementations returned identical bytes and identical exit codes for')
    print('every row above. The only difference between them here is speed.')
else:
    print('The implementations returned different results. Ignore every')
    print('recommendation below.')
    for row in results:
        if row.python.digests != row.rust.digests:
            print(f'     different output: {row.label}')
        if row.python.codes != row.rust.codes:
            print(f'     different exit status: {row.label}')

if unsteady:
    print('\nA command did not return the same result on every repetition, so a file')
    print('changed while the benchmark was running. The timings are still usable, but')
    print('these rows are worth measuring again:')
    for label, impl in unsteady:
        print(f'     {impl:<7} {label}')
elif not CLEAN:
    print('\nEvery repetition of one row exited with a non-zero status, identically in')
    print('both implementations. The tool reported a file it could not read. That is a')
    print('fact about the files, not a difference between the implementations.')
    for row in results:
        codes = row.python.codes | row.rust.codes
        if codes != frozenset({0}):
            print(f'     {row.label}: exit {sorted(codes)}')

if noisy:
    print(f'\nIn these rows the median is more than {SPREAD_LIMIT:.0f}% above the')
    print('minimum, which means the machine was busy. Measure them again:')
    for label, impl, percent in noisy:
        print(f'     {impl:<7} {label:<44} +{percent:.0f}%')
else:
    print(f'Every median is within {SPREAD_LIMIT:.0f}% of its minimum.')

| what is being processed | Python min / median | Rust min / median | Python is |
| -- | --: | --: | --: |
| one file | 26.1 / 27.0 ms | 1.4 / 1.5 ms | ≥18.0x slower |
| a typical commit: 5 files | 26.6 / 27.5 ms | 1.5 / 1.6 ms | ≥18.0x slower |
| this repository: 6 prose files at 30 KB each | 62.8 / 64.0 ms | 3.7 / 4.0 ms | 17.0x slower |
| a large repository: 2000 files | 201.5 / 207.7 ms | 33.9 / 35.8 ms | 5.9x slower |
| one 4 MB document | 1059.6 / 1070.3 ms | 70.9 / 71.2 ms | 14.9x slower |

≥ means the Rust time is within 2 times the 1.10 ms floor, so most of what it measures is the cost of starting any process at all. Those ratios are lower bounds, and the real difference is larger.

Both implementations returned identical bytes and identical exit codes for
every row above. The only difference between them here is speed.
Every median is within 15% of its minimum.


## What the time is spent on

The table above combines two separate costs: a fixed cost paid once per run,
and a cost paid for each file. Telling them apart is what makes it possible to
recommend one implementation per way of using the tool.

The next cell measures the fixed cost directly. It times the cheapest process
this machine can start, a Python interpreter that does nothing, the same
interpreter after importing this tool, and then each of the two programs on a
single file.

The difference between the two interpreter rows is the cost of importing the
tool. The difference between the last two rows is what one run of the Rust
program saves. The first row is the cost that neither implementation can go
below, because it is what starting any process costs.

In [5]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure

# Element ids are derived from this value rather than from memory addresses, so
# two runs over the same numbers produce the same file.
mpl.rcParams['svg.hashsalt'] = 'markdown-prose-hooks'


def hide_spines(*all_axes: Axes) -> None:
    """Remove the top and right borders from each chart."""
    for axes in all_axes:
        axes.spines['top'].set_visible(False)
        axes.spines['right'].set_visible(False)


def save_chart(figure: Figure, name: str) -> None:
    """Write ``figure`` to the docs directory as the bytes that get committed.

    Two things would otherwise change the file on every run, whatever was
    measured. matplotlib writes the current date into the SVG metadata, and it
    leaves trailing spaces on a few hundred lines, which this repository
    removes with its own hook when the file is committed. Left as they are, the
    committed chart is not the file this notebook wrote, running the notebook
    always leaves uncommitted changes, and a real change to a measurement is
    hard to see among the changes that mean nothing.
    """
    path = REPO / 'docs' / name
    figure.savefig(path, metadata={'Date': None})
    plt.close(figure)
    text = path.read_text(encoding='utf-8')
    path.write_text(
        chr(10).join(line.rstrip() for line in text.split(chr(10))),
        encoding='utf-8',
    )
    print(f'chart written to docs/{name}')


floor_rows = {
    'cheapest process': FLOOR,
    'Python interpreter, doing nothing': measure([sys.executable, '-c', 'pass']),
    'Python interpreter, after import': measure(
        [sys.executable, '-c', 'import markdown_prose_hooks.unwrap']
    ),
    'Python program, one file': measure(
        [str(PYTHON_CLI), str(scratch / 'tiny.md'), '--json']
    ),
    'Rust program, one file': measure(
        [str(RUST_CLI), str(scratch / 'tiny.md'), '--json']
    ),
}
for label, timing in floor_rows.items():
    print(f'{label:<36} {timing.min:>6.2f} ms   (median {timing.median:>6.2f} ms)')

python_startup = floor_rows['Python program, one file'].min
rust_startup = floor_rows['Rust program, one file'].min
saving = python_startup - rust_startup

print(f'\nPython takes {python_startup / rust_startup:.1f} times as long to start')
if rust_startup > SPAWN_FLOOR:
    corrected = (python_startup - SPAWN_FLOOR) / (rust_startup - SPAWN_FLOOR)
    print(f'With the floor removed from both, it would be {corrected:.1f} times.')
    print(
        f'   That figure is shown but not used. The Rust time is only '
        f'{rust_startup / SPAWN_FLOOR:.1f} times'
    )
    print('   the floor, so removing the floor divides by the small difference between')
    print('   two similar numbers, and the result changes greatly when the floor')
    print('   changes slightly. Treat the figure above as a minimum instead.')
else:
    print('   The Rust program was as fast as the cheapest process this machine can')
    print('   start, so its own startup cost is below what this method can measure.')
    print('   The figure above is a minimum.')

print(f'\nEach run of the Rust program saves {saving:.1f} ms.')
print(f'Over a hundred runs a day that is {saving * 100 / 1000:.1f} seconds.')
print('   Subtracting is reliable here even though dividing is not. The cost of')
print('   starting a process is the same in both numbers, so it cancels out when')
print('   they are subtracted.')

figure, axes = plt.subplots(figsize=(8, 3.2))
labels = list(floor_rows)
axes.barh(
    labels,
    [floor_rows[label].min for label in labels],
    color=['#999999', '#bbbbbb', '#bbbbbb', '#1f77b4', '#ff7f0e'],
)
axes.axvline(SPAWN_FLOOR, color='#444444', linestyle='--', linewidth=1)
axes.set_xscale('log')
axes.set_xlabel('time (milliseconds, logarithmic scale)')
axes.set_title('The dashed line is what starting any process costs')
axes.invert_yaxis()
for index, label in enumerate(labels):
    axes.text(
        floor_rows[label].min * 1.15,
        index,
        f'{floor_rows[label].min:.2f} ms',
        va='center',
        fontsize=9,
    )
axes.set_xlim(right=max(t.min for t in floor_rows.values()) * 3)
axes.grid(alpha=0.3, axis='x')
hide_spines(axes)
figure.tight_layout()
print()
save_chart(figure, 'benchmarks-startup.svg')

cheapest process                       1.10 ms   (median   1.15 ms)
Python interpreter, doing nothing     12.70 ms   (median  13.23 ms)
Python interpreter, after import      25.56 ms   (median  26.78 ms)
Python program, one file              29.23 ms   (median  31.36 ms)
Rust program, one file                 1.61 ms   (median   1.93 ms)

Python takes 18.1 times as long to start
With the floor removed from both, it would be 54.4 times.
   That figure is shown but not used. The Rust time is only 1.5 times
   the floor, so removing the floor divides by the small difference between
   two similar numbers, and the result changes greatly when the floor
   changes slightly. Treat the figure above as a minimum instead.

Each run of the Rust program saves 27.6 ms.
Over a hundred runs a day that is 2.8 seconds.
   Subtracting is reliable here even though dividing is not. The cost of
   starting a process is the same in both numbers, so it cancels out when
   they are subtracted.



chart written to docs/benchmarks-startup.svg


![The dashed line is what starting any process costs](benchmarks-startup.svg)

## Separating the fixed cost from the cost per file

Running the tool once over an increasing number of files separates the two
costs. In the left chart, where a line begins is the fixed cost, and how
steeply it rises is the cost per file. The right chart shows how many times
slower Python is at each number of files.

Both axes on the left are logarithmic because the number of files spans three
orders of magnitude. On ordinary axes, six of the eight measurements would fall
within the first quarter of the chart, where the fixed cost cannot be read.

In [6]:
# Fewer repetitions than the table above. This cell and the next one exist to
# show the shape of a curve rather than to produce a figure quoted elsewhere,
# and the shape is already stable at this number of repetitions.
SWEEP_REPS = 15

counts = 1, 10, 50, 100, 250, 500, 1000, 2000
curve = {'Python': [], 'Rust': []}
for count in counts:
    args = file_list(count)
    curve['Python'].append(measure([str(PYTHON_CLI), *args, '--json'], SWEEP_REPS).min)
    curve['Rust'].append(measure([str(RUST_CLI), *args, '--json'], SWEEP_REPS).min)

figure, (left, right) = plt.subplots(1, 2, figsize=(11, 4.2))
for name, series in curve.items():
    left.plot(counts, series, marker='o', label=name)
left.axhline(SPAWN_FLOOR, color='#444444', linestyle='--', linewidth=1)
left.text(counts[0], SPAWN_FLOOR * 1.15, 'floor', fontsize=8, color='#444444')
left.set_xscale('log')
left.set_yscale('log')
left.set_xlabel('files processed in one run')
left.set_ylabel('time (milliseconds)')
left.set_title('Where a line starts is the fixed cost')
left.legend()
left.grid(alpha=0.3, which='both')

ratios = [p / r for p, r in zip(curve['Python'], curve['Rust'], strict=True)]
right.plot(counts, ratios, marker='o', color='#2ca02c')
right.set_xscale('log')
right.set_ylim(bottom=0)
right.set_xlabel('files processed in one run')
right.set_ylabel('times slower')
right.set_title('Python, relative to Rust')
right.grid(alpha=0.3, which='both')
hide_spines(left, right)

# The chart is written next to the notebook rather than embedded in it, and as
# SVG rather than PNG. An embedded PNG is stored as base64 text, which the two
# spell-checking hooks read as prose: measured on thirty different versions of
# this chart, codespell reported a misspelling in eighteen of them, because
# base64 splits into thousands of short letter sequences and some of them match
# dictionary entries. The same thirty charts written as SVG produced no
# findings from either hook. A PNG file would be worse still, because this
# repository routes `*.png` through Git LFS, so the chart would be committed as
# a pointer and appear broken to anyone cloning without git-lfs installed.
figure.tight_layout()
save_chart(figure, 'benchmarks.svg')

# A straight line between the first and last points is only a fair summary if
# the measurements between them are also close to that line. Every consecutive
# pair is printed so that a reader can see whether they are, rather than being
# asked to assume it.
count_marginal = {}
print(f'{"":<8} {"fixed cost":>11} {"per file":>12}   cost per file between each pair')
for name, series in curve.items():
    pairs = [
        (series[i + 1] - series[i]) / (counts[i + 1] - counts[i]) * 1000
        for i in range(len(counts) - 1)
    ]
    count_marginal[name] = (series[-1] - series[0]) / (counts[-1] - counts[0]) * 1000
    print(
        f'{name:<8} {series[0]:>8.1f} ms {count_marginal[name]:>8.1f} us   '
        + ' '.join(f'{value:.0f}' for value in pairs)
    )
print('\nThe leftmost pairs are the least reliable, because they are differences')
print('between measurements only a few milliseconds apart. They stabilize once the')
print('per-file work is large enough to measure, and that is what makes the single')
print('figure in the third column a fair summary rather than a straight line drawn')
print('through a curve.')
print(
    f'\nPython is {ratios[0]:.1f} times slower at {counts[0]} file and '
    f'{ratios[-1]:.1f} times slower at {counts[-1]} files.'
)

chart written to docs/benchmarks.svg
          fixed cost     per file   cost per file between each pair
Python       30.2 ms     86.1 us   95 55 167 62 103 85 83
Rust          1.9 ms     16.3 us   57 8 27 10 18 15 17

The leftmost pairs are the least reliable, because they are differences
between measurements only a few milliseconds apart. They stabilize once the
per-file work is large enough to measure, and that is what makes the single
figure in the third column a fair summary rather than a straight line drawn
through a curve.

Python is 15.8 times slower at 1 file and 5.9 times slower at 2000 files.


![Where a line starts is the fixed cost](benchmarks.svg)

## What the cost per file depends on

The cost of one more file is not a fixed number, and it is not a property of
the tool. Opening a file is a request to the operating system, and costs about
the same in both languages. Reading its contents and rewriting the paragraphs
is the work the two implementations do differently. So how much slower Python
is per file depends on how much text an average file contains.

The next cell measures that. For each file size it times two runs, one over a
small number of files and one over a large number, and subtracts the first from
the second. Subtracting removes the fixed startup cost exactly, without having
to estimate it. That matters most for small files, where startup would
otherwise be most of what was measured.

The result is a single curve. At one end, processing a file costs little more
than opening it. At the other end, the cost is almost entirely the work of
rewriting the text. Any figure quoted for the difference per file is one point
on this curve, and which point applies to a repository depends on the
Markdown files in it. The marked point is the value for this repository.

In [7]:
import numpy as np

# Two numbers of files for each file size, and the difference between them. The
# fixed startup cost is identical in both, so subtracting removes it exactly
# and nothing here needs an estimate of it.
SIZE_REPS = 8
LOW, HIGH = 100, 500
paragraph_counts = 1, 3, 10, 30, 100


def listing(folder: Path, count: int) -> list[str]:
    """Return arguments naming ``count`` files from ``folder``, asking for JSON."""
    path = scratch / f'{folder.name}-{count}.txt'
    path.write_text(
        '\n'.join(str(folder / f'{index}.md') for index in range(count)) + '\n'
    )
    return ['--files-from', str(path), '--json']


def marginal_us(cli: Path, folder: Path) -> float:
    """Return the microseconds one more file from ``folder`` costs ``cli``.

    The fixed startup cost is identical in both measurements, so subtracting
    one from the other removes it exactly rather than by estimate.
    """
    low = measure([str(cli), *listing(folder, LOW)], SIZE_REPS).min
    high = measure([str(cli), *listing(folder, HIGH)], SIZE_REPS).min
    return (high - low) / (HIGH - LOW) * 1000


sweep = {'bytes': [], 'Python': [], 'Rust': []}
for paragraphs in paragraph_counts:
    folder = scratch / f'size-{paragraphs}'
    folder.mkdir(exist_ok=True)
    for index in range(HIGH):
        (folder / f'{index}.md').write_text(PARAGRAPH * paragraphs)
    sweep['bytes'].append(len(PARAGRAPH) * paragraphs)
    sweep['Python'].append(marginal_us(PYTHON_CLI, folder))
    sweep['Rust'].append(marginal_us(RUST_CLI, folder))

size_ratios = [p / r for p, r in zip(sweep['Python'], sweep['Rust'], strict=True)]

# Interpolated on a logarithmic scale, because the sizes measured are spaced
# logarithmically.
REPO_RATIO = float(
    np.interp(np.log10(REPO_MEAN_BYTES), np.log10(sweep['bytes']), size_ratios)
)
CLAMPED = not sweep['bytes'][0] <= REPO_MEAN_BYTES <= sweep['bytes'][-1]
RATIO_TEXT = f'{REPO_RATIO:.1f} times or more' if CLAMPED else f'{REPO_RATIO:.1f} times'

# Kept within the range measured, so that a mean larger than the largest size
# measured is marked at the end of the curve rather than stretching the chart
# into empty space.
marker_x = min(max(REPO_MEAN_BYTES, sweep['bytes'][0]), sweep['bytes'][-1])

figure, axes = plt.subplots(figsize=(8, 4.5))
axes.plot(sweep['bytes'], size_ratios, marker='o', color='#2ca02c')
axes.axvline(marker_x, color='#d62728', linestyle='--', linewidth=1)
axes.annotate(
    f'this repository\n{REPO_MEAN_BYTES:,.0f} bytes per file, {RATIO_TEXT}',
    xy=(marker_x, REPO_RATIO),
    xytext=(-16, -78) if CLAMPED else (16, -34),
    textcoords='offset points',
    horizontalalignment='right' if CLAMPED else 'left',
    fontsize=9,
    color='#d62728',
    arrowprops={'arrowstyle': '->', 'color': '#d62728'},
)
axes.set_xscale('log')
axes.set_ylim(bottom=0)
axes.set_xlabel(
    f'bytes per file (logarithmic scale; each point subtracts {LOW} files from {HIGH})'
)
axes.set_ylabel('times slower, per file')
axes.set_title('How much slower Python is per file depends on the text in it')
axes.grid(alpha=0.3, which='both')
hide_spines(axes)
figure.tight_layout()
save_chart(figure, 'benchmarks-bytes.svg')

table = [
    '| bytes per file | Python | Rust | Python is |',
    '| --: | --: | --: | --: |',
]
for size, python_us, rust_us, value in zip(
    sweep['bytes'], sweep['Python'], sweep['Rust'], size_ratios, strict=True
):
    table.append(
        f'| {size} | {python_us:.1f} us per file | {rust_us:.1f} us per file '
        f'| {value:.1f}x slower |'
    )
table.append('')
table.append(
    f'Across the {sweep["bytes"][-1] // sweep["bytes"][0]}-fold range of file sizes '
    f'measured, Python is between {min(size_ratios):.1f} and {max(size_ratios):.1f} '
    f'times slower per file. Files in this repository average '
    f'{REPO_MEAN_BYTES:,.0f} bytes, where the figure is **{RATIO_TEXT}**'
    + (
        '. That is beyond the largest size measured, but the curve has already '
        'leveled off by then, so the true figure is close to the value at the end '
        'of the curve rather than far above it.'
        if CLAMPED
        else '.'
    )
)
display(Markdown(chr(10).join(table)))

# The previous cell measured this same quantity at one file size, because the
# files it used are 207 bytes each. Two independent measurements of one
# quantity are worth comparing rather than assuming they agree.
common = sweep['bytes'].index(len(PARAGRAPH) * 3)
print(
    f'Checked against the previous cell at {sweep["bytes"][common]} bytes per file, '
    'where both measured the same thing:'
)
for name in ('Python', 'Rust'):
    here, there = sweep[name][common], count_marginal[name]
    print(
        f'  {name:<7} {here:>6.1f} us here, {there:>6.1f} us there, a difference of '
        f'{abs(here - there) / there * 100:.0f}%'
    )

chart written to docs/benchmarks-bytes.svg


| bytes per file | Python | Rust | Python is |
| --: | --: | --: | --: |
| 69 | 44.7 us per file | 13.3 us per file | 3.4x slower |
| 207 | 88.4 us per file | 15.6 us per file | 5.7x slower |
| 690 | 213.5 us per file | 23.5 us per file | 9.1x slower |
| 2070 | 550.2 us per file | 48.8 us per file | 11.3x slower |
| 6900 | 1745.3 us per file | 128.7 us per file | 13.6x slower |

Across the 100-fold range of file sizes measured, Python is between 3.4 and 13.6 times slower per file. Files in this repository average 30,395 bytes, where the figure is **13.6 times or more**. That is beyond the largest size measured, but the curve has already leveled off by then, so the true figure is close to the value at the end of the curve rather than far above it.

Checked against the previous cell at 207 bytes per file, where both measured the same thing:
  Python    88.4 us here,   86.1 us there, a difference of 3%
  Rust      15.6 us here,   16.3 us there, a difference of 5%


![How much slower Python is per file depends on the text in it](benchmarks-bytes.svg)

## How to choose

The cell below builds a recommendation from the measurements above and from the
current contents of the repository. It reads `action.yml` to find which
implementation the GitHub Action runs today, and checks whether a release
exists, because some of what it reports can only change once one does.

In [8]:
import textwrap

action = (REPO / 'action.yml').read_text()
ACTION_USES_BINARY = 'setup-python' not in action
RELEASED = bool(TAGS)

startup_ratio = python_startup / rust_startup
bound = BOUND if rust_startup < SPAWN_FLOOR * FLOOR_FACTOR else ''
transform_ratio = max(size_ratios)

# Each configuration file that cannot yet take its intended form is marked with
# this phrase, and the comment below the marker says what is blocked and what it
# is blocked on. Those comments are read here rather than summarized, because a
# list of the same caveats kept in this notebook would be a second copy to
# maintain, and the copy nobody edits is the one that becomes wrong.
MARKER = 'DEVIATION, blocked on'


def deviation(path: Path) -> str:
    """Return the comment block introduced by ``MARKER`` in ``path``, as one line."""
    # The only read in this notebook that named no encoding, which meant it
    # decoded with whatever the platform default happened to be.
    lines = path.read_text(encoding='utf-8').splitlines()
    for index, line in enumerate(lines):
        if MARKER in line:
            block = []
            for follow in lines[index:]:
                stripped = follow.lstrip()
                if not stripped.startswith('#'):
                    break
                block.append(stripped.lstrip('#').strip())
            return ' '.join(part for part in block if part)
    return ''


pending = []
for name in ('action.yml', '.pre-commit-hooks.yaml', 'pyproject.toml', 'Cargo.toml'):
    note = deviation(REPO / name)
    if note:
        # The first sentence is the marker itself, which the heading already
        # says. The two after it describe the deviation.
        sentences = note.split('. ')[1:3]
        pending.append(f'**`{name}`** -- ' + '. '.join(sentences).rstrip('.') + '.')

if ACTION_USES_BINARY:
    action_why = (
        'There is nothing to choose. The action selects an implementation itself, and '
        f'it downloads a prebuilt {BINARY_KB:.0f} KB executable, which installs faster '
        'than either language toolchain and then runs faster as well.'
    )
else:
    action_why = (
        'There is nothing to choose. The action installs Python and runs the Python '
        'program, and it takes no input that would change this. '
        + (
            'A release now exists, so it could be changed to download a prebuilt '
            'executable instead.'
            if RELEASED
            else 'Downloading a prebuilt executable instead requires a release to '
            'download it from, and no release exists yet.'
        )
    )

cli_install = (
    'Install it with `pipx install markdown-prose-hooks`.'
    if RELEASED
    else 'For now it is installed from a checkout. Nothing is published to PyPI until '
    'the first release.'
)
if not RELEASED:
    pending.append(
        '**package registries** -- neither PyPI nor crates.io has this package '
        'yet, so both command-line rows describe installing from a checkout.'
    )

rows = [
    (
        '`pre-commit`, ordinary repository',
        '`-py`',
        f'Almost all of the cost is starting the program: {bound}{startup_ratio:.0f} '
        f'times, or {saving:.0f} ms per run. `pre-commit` is itself a Python '
        'application, so the interpreter is already installed. The Rust hook is '
        'compiled from source and needs cargo.',
    ),
    (
        '`pre-commit`, large repository, or `--all-files`',
        '`-rs`',
        'Checking every file makes the cost per file the one that matters, and that '
        f'cost depends on file size. Python is {min(size_ratios):.1f} times slower per '
        f'file when files are nearly empty and {transform_ratio:.0f} times slower once '
        'they contain several thousand bytes of text. Files here average '
        f'{REPO_MEAN_BYTES:,.0f} bytes, '
        f'where it is {RATIO_TEXT} slower. The Rust hook is compiled once and reused '
        'afterwards.',
    ),
    ('GitHub Action', 'neither', action_why),
    (
        'A script or CI step without Python',
        '`-rs`',
        'A single executable file. No language runtime has to be installed first.',
    ),
    ('Command line, with Python already installed', '`-py`', cli_install),
]

document = ['| how you run it | which to use | why |', '| -- | -- | -- |']
document += [f'| {channel} | {use} | {why} |' for channel, use, why in rows]

document += [
    '',
    '### How much does this matter in practice?',
    '',
    textwrap.fill(
        f'Each run saves {saving:.0f} ms, which is about '
        f'{saving * 100 / 1000:.1f} seconds over a hundred runs a day. That is a small '
        'number. On its own it is not a reason to add a Rust toolchain to a project '
        'that does not already have one.',
        width=86,
    ),
    '',
    textwrap.fill(
        'The saving is larger whenever the cost per file matters more than the cost of '
        'starting the program. That happens when one run processes thousands of files: '
        '`pre-commit run --all-files` in a large repository, or a CI job that checks '
        f'the whole tree. For files the size this repository holds, Python takes '
        f'{RATIO_TEXT} as long as Rust for each file it processes.',
        width=86,
    ),
]

if pending:
    document += [
        '',
        '### Not yet possible, pending the first release',
        '',
        textwrap.fill(
            'What this notebook measured is what the repository does today. Below are '
            'the places where that differs from what it is designed to do, because no '
            'release exists yet. Each note is quoted from the file that records it.',
            width=86,
        ),
        '',
    ]
    document += [f'- {note}' for note in pending]

if not AGREE:
    document += [
        '',
        '**The implementations returned different results above. Ignore all of this.**',
    ]

display(Markdown(chr(10).join(document)))

| how you run it | which to use | why |
| -- | -- | -- |
| `pre-commit`, ordinary repository | `-py` | Almost all of the cost is starting the program: ≥18 times, or 28 ms per run. `pre-commit` is itself a Python application, so the interpreter is already installed. The Rust hook is compiled from source and needs cargo. |
| `pre-commit`, large repository, or `--all-files` | `-rs` | Checking every file makes the cost per file the one that matters, and that cost depends on file size. Python is 3.4 times slower per file when files are nearly empty and 14 times slower once they contain several thousand bytes of text. Files here average 30,395 bytes, where it is 13.6 times or more slower. The Rust hook is compiled once and reused afterwards. |
| GitHub Action | neither | There is nothing to choose. The action installs Python and runs the Python program, and it takes no input that would change this. Downloading a prebuilt executable instead requires a release to download it from, and no release exists yet. |
| A script or CI step without Python | `-rs` | A single executable file. No language runtime has to be installed first. |
| Command line, with Python already installed | `-py` | For now it is installed from a checkout. Nothing is published to PyPI until the first release. |

### How much does this matter in practice?

Each run saves 28 ms, which is about 2.8 seconds over a hundred runs a day. That is a
small number. On its own it is not a reason to add a Rust toolchain to a project that
does not already have one.

The saving is larger whenever the cost per file matters more than the cost of starting
the program. That happens when one run processes thousands of files: `pre-commit run
--all-files` in a large repository, or a CI job that checks the whole tree. For files
the size this repository holds, Python takes 13.6 times or more as long as Rust for
each file it processes.

### Not yet possible, pending the first release

What this notebook measured is what the repository does today. Below are the places
where that differs from what it is designed to do, because no release exists yet. Each
note is quoted from the file that records it.

- **`action.yml`** -- This provisions Python and runs the `-py` implementation, which is not the intended shape. The design calls for downloading a prebuilt binary from the release for this tag -- roughly a megabyte, no toolchain, faster to install as well as to run -- with an `implementation` input taking `auto`, `rust` or `python`, and `pip install` kept as the fallback for a runner with no published binary.
- **`.pre-commit-hooks.yaml`** -- This file is meant to go away. At `v0.1.0` each pair moves to a mirror repository serving only its own implementation -- `markdown-prose-hooks-py` and `markdown-prose-hooks-rs` -- so a consumer stops cloning roughly 1.2 MB carrying both plus 355 corpus fixtures to get one of them.
- **package registries** -- neither PyPI nor crates.io has this package yet, so both command-line rows describe installing from a checkout.